# FORTRAN - praca z plikami
Szósty blok zajęć: zapis/odczyt danych i bezpieczna obsługa plików.

Cele zajęć:
- rozumieć `open`, `read`, `write`, `close`
- używać `iostat` do wykrywania błędów
- formatować dane przy zapisie do pliku
- pracować z plikami o nieznanym z góry rozmiarze

Plan spotkania:
1. Konwersja typów przez wewnętrzne `read`/`write`
2. Podstawowy zapis i odczyt pliku
3. Odczyt pliku o nieznanej liczbie wierszy
4. Formatowany zapis raportu
5. Mini-checkpoint + ćwiczenia + ściąga

Materiał pomocniczy:
- https://www.tutorialspoint.com/fortran/fortran_file_input_output.htm


## 0) Konwersja typów przez tekst (internal file I/O)

W Fortranie można użyć zmiennej `character` jako źródła lub celu `read`/`write`.


In [ ]:
program konwersja_typu
  implicit none

  real :: liczba = 5.125
  character(len=20) :: napis = ""
  real :: odczyt

  write(napis, '(f8.3)') liczba
  print *, "real -> character:", trim(napis)

  napis = "15.089"
  read(napis, *) odczyt
  print *, "character -> real:", odczyt
end program konwersja_typu


## 1) Podstawowy zapis i odczyt pliku

Dobra praktyka:
- przy zapisie testowym używaj `status='replace'`,
- sprawdzaj `iostat`,
- zawsze zamykaj plik (`close`).


In [ ]:
program zapis_i_odczyt
  implicit none

  real :: x(10), y(10), p(10), q(10)
  integer :: i, ios

  do i = 1, 10
     x(i) = i * 0.1
     y(i) = sin(x(i))
  end do

  open(10, file='data1.dat', status='replace', action='write', iostat=ios)
  if (ios /= 0) stop "Nie mozna otworzyc pliku do zapisu"

  do i = 1, 10
     write(10,'(f8.3,1x,f10.6)') x(i), y(i)
  end do
  close(10)

  open(11, file='data1.dat', status='old', action='read', iostat=ios)
  if (ios /= 0) stop "Nie mozna otworzyc pliku do odczytu"

  do i = 1, 10
     read(11,*,iostat=ios) p(i), q(i)
     if (ios /= 0) exit
  end do
  close(11)

  print *, "Pierwsze 3 wiersze:"
  do i = 1, 3
     write(*,'(f8.3,1x,f10.6)') p(i), q(i)
  end do
end program zapis_i_odczyt


## 2) Odczyt pliku o nieznanej liczbie wierszy

Typowy schemat:
1. Pierwszy przebieg: policz wiersze (`iostat` + `exit`).
2. Alokuj tablice.
3. Drugi przebieg: wczytaj dane.


In [ ]:
program plik_nieznany_rozmiar
  implicit none

  real, allocatable :: jd(:), js(:), ejs(:)
  integer :: n, io, i

  n = 0
  open(20, file='dane.dat', status='old', action='read')
  do
     read(20,*,iostat=io)
     if (io /= 0) exit
     n = n + 1
  end do
  close(20)

  allocate(jd(n), js(n), ejs(n))

  open(20, file='dane.dat', status='old', action='read')
  do i = 1, n
     read(20,*) jd(i), js(i), ejs(i)
  end do
  close(20)

  print *, "Wierszy:", n
  print *, "Srednia jasnosc:", sum(js)/real(n)

  deallocate(jd, js, ejs)
end program plik_nieznany_rozmiar


## 3) Formatowany zapis wyników

Stały format bardzo ułatwia późniejsze wykresy i analizę.


In [ ]:
program zapis_formatowany
  implicit none

  real :: t, y
  integer :: i

  open(30, file='raport.dat', status='replace', action='write')
  write(30,'(a)') "# t[s]    y"

  do i = 0, 10
     t = real(i)
     y = exp(-t/5.0)
     write(30,'(f8.2,1x,f10.6)') t, y
  end do

  close(30)
  print *, "Zapisano plik raport.dat"
end program zapis_formatowany


### Mini-checkpoint

1. Czemu warto używać `status='replace'` podczas testów?
2. Co oznacza `iostat /= 0` przy `read`?
3. Po co robić dwa przebiegi przy pliku o nieznanej długości?

<details>
<summary>Odpowiedzi</summary>

- Żeby program nadpisywał plik bez błędu „plik już istnieje”.
- Wystąpił błąd lub koniec pliku.
- Najpierw trzeba znać liczbę rekordów, by poprawnie zaalokować tablice.

</details>


## 4) Typowe błędy początkujących

### 1) `status='new'` przy każdym uruchomieniu
Drugie uruchomienie kończy się błędem, jeśli plik już istnieje.

### 2) Brak `close(unit)`
Może prowadzić do niepełnego zapisu danych.

### 3) Brak obsługi `iostat`
Program „idzie dalej”, mimo że odczyt się nie udał.


## 5) Ćwiczenia

1. Wczytaj `lightcurve.dat`, policz średnią i zapisz plik z wartościami odjętymi o średnią.
2. Zapisz tabelę `x, sin(x), cos(x)` dla `x=0..2*pi`.
3. Dodaj nagłówek z datą i liczbą punktów do pliku wynikowego.


## 6) Ściąga

- Otwieranie: `open(unit, file='nazwa', status='old|replace', action='read|write')`
- Odczyt: `read(unit, fmt, iostat=ios) ...`
- Zapis: `write(unit, fmt) ...`
- Zamknięcie: `close(unit)`
- Wewnętrzne I/O: `write(tekst,'(...)') liczba`, `read(tekst,*) liczba`
